In [1]:
import os

import gymnasium
import numpy as np
import torch
import yaml
from PIL import Image
from torchvision.transforms import v2
from einops import rearrange

from config import METAWORLD_CFGS, DMC_CFGS
from metaworld_env import setup_metaworld_env
from dmc import setup_dmc_env

from model.encoder import EncoderNet
from model.actor import DDPGActorNet
from model.critic import CriticNet

In [2]:
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write("""{
        "file_format_version" : "1.0.0",
        "ICD" : {
            "library_path" : "libEGL_nvidia.so.0"
        }
    }
    """)
        
os.environ["MUJOCO_GL"] = "egl"

In [3]:
task_name = "hopper"
seed = 0

In [4]:
def setup_env(task_name:str, seed:int, render_mode:str|None = None) -> gymnasium.Env:
    if task_name in METAWORLD_CFGS:
        env = setup_metaworld_env(task_name, seed, render_mode)
    elif task_name in DMC_CFGS:
        env = setup_dmc_env(task_name, seed, render_mode)
    env = gymnasium.wrappers.RecordEpisodeStatistics(env)
    env = gymnasium.wrappers.AddRenderObservation(env, render_only=True)
    env = gymnasium.wrappers.FrameStackObservation(env, 2)

    return env

In [5]:
if task_name in METAWORLD_CFGS:
    config_path = "configs/ddpg_metaworld.yaml"
elif task_name in DMC_CFGS:
    config_path = "configs/ddpg_dmc.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
envs = setup_env(task_name, 5000, "rgb_array")

obs_dim = envs.observation_space.shape
action_dim = envs.action_space.shape

In [8]:
def rollout():
    obs, info = envs.reset()
    success = 0
    for i in range(500):
        action = envs.action_space.sample()
        next_obs, reward, done, timeout, info = envs.step(action[0])
        #success += info["success"]
        obs = next_obs

In [10]:
%%timeit 
rollout()

1.1 s ± 8.21 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
